In [19]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('data/product_catalog.csv')

# Basic info
print("Shape:", df.shape)
print("\nColumn Names:\n", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nFirst Row:\n", df.head(1))

df.isnull().sum()

Shape: (5368, 18)

Column Names:
 ['sku_id', 'product_name', 'category', 'subcategory', 'brand', 'base_price_usd', 'cost_price_usd', 'current_price_usd', 'min_price_usd', 'max_price_usd', 'inventory_count', 'restock_days', 'avg_rating', 'review_count', 'weight_kg', 'is_active', 'launch_date', 'tags']

Data Types:
 sku_id                object
product_name          object
category              object
subcategory           object
brand                 object
base_price_usd       float64
cost_price_usd       float64
current_price_usd    float64
min_price_usd        float64
max_price_usd        float64
inventory_count        int64
restock_days         float64
avg_rating           float64
review_count           int64
weight_kg            float64
is_active               bool
launch_date           object
tags                  object
dtype: object

First Row:
       sku_id                          product_name     category  subcategory  \
0  SKU001000  NovaTech Ultra Smartphones White 792  Ele

sku_id                  0
product_name            0
category                0
subcategory             0
brand                   0
base_price_usd          0
cost_price_usd          0
current_price_usd       0
min_price_usd           0
max_price_usd           0
inventory_count         0
restock_days         3658
avg_rating              0
review_count            0
weight_kg               0
is_active               0
launch_date             0
tags                    0
dtype: int64

In [9]:
# =========================
# DATA PREPROCESSING
# =========================

# Convert date
df['launch_date'] = pd.to_datetime(df['launch_date'], errors='coerce')

# Fill missing values
df['restock_days'].fillna(df['restock_days'].median(), inplace=True)

# Convert boolean
df['is_active'] = df['is_active'].astype(int)

# Convert tags (list string → count feature)
import ast
df['tags_count'] = df['tags'].apply(lambda x: len(ast.literal_eval(x)) if pd.notnull(x) else 0)

# Drop unnecessary columns
df.drop(['sku_id', 'product_name', 'tags'], axis=1, inplace=True)

# Encode categorical columns
from sklearn.preprocessing import LabelEncoder

cat_cols = ['category', 'subcategory', 'brand']
le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

/var/folders/rb/wmvx_2cs4pb2l6mm945ttmv00000gn/T/ipykernel_14220/1156968773.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['restock_days'].fillna(df['restock_days'].median(), inplace=True)


In [10]:
# Features & Target
X = df.drop('current_price_usd', axis=1)
y = df['current_price_usd']

# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate(model):
    pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)
    
    return mae, mse, rmse, r2

In [16]:
df.drop('launch_date', axis=1, inplace=True)

In [17]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor()
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = evaluate(model)

# Show results
for name, scores in results.items():
    print(f"\n{name}")
    print(f"MAE : {scores[0]}")
    print(f"MSE : {scores[1]}")
    print(f"RMSE: {scores[2]}")
    print(f"R2  : {scores[3]}")

DTypePromotionError: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Int64DType'>)

In [21]:
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv('data/product_catalog.csv')

# Copy for recommendation
df_rec = df.copy()

# =========================
# 2. DATA PREPROCESSING
# =========================

# ❌ Drop useless column (100% NaN)
df.drop(['restock_days'], axis=1, inplace=True)

# -------------------------
# Date Handling
# -------------------------
df['launch_date'] = pd.to_datetime(df['launch_date'], format='%d/%m/%y', errors='coerce')
df['launch_year'] = df['launch_date'].dt.year
df['launch_year'] = df['launch_year'].fillna(df['launch_year'].median())
df['launch_year'] = df['launch_year'].fillna(0)
df.drop(['launch_date'], axis=1, inplace=True)

# -------------------------
# Boolean Handling
# -------------------------
df['is_active'] = df['is_active'].map({'TRUE': 1, 'FALSE': 0, True: 1, False: 0})
df['is_active'] = df['is_active'].fillna(0)

# -------------------------
# Tags → count
# -------------------------
def count_tags(tag_str):
    try:
        return len(ast.literal_eval(tag_str))
    except:
        return 0

df['tags_count'] = df['tags'].apply(count_tags)

# -------------------------
# Feature Engineering 🔥
# -------------------------
df['price_margin'] = df['base_price_usd'] - df['cost_price_usd']
df['inventory_value'] = df['inventory_count'] * df['cost_price_usd']
df['rating_weighted'] = df['avg_rating'] * df['review_count']

# -------------------------
# Encode categorical
# -------------------------
cat_cols = ['category', 'subcategory', 'brand']

for col in cat_cols:
    df[col] = df[col].fillna("Unknown")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# =========================
# 3. FEATURE & TARGET
# =========================

# ❌ REMOVE DATA LEAKAGE FEATURES
X = df.drop([
    'sku_id',
    'product_name',
    'tags',
    'current_price_usd',
    'base_price_usd',
    'min_price_usd',
    'max_price_usd'
], axis=1)

y = df['current_price_usd']

# -------------------------
# Handle remaining NaNs
# -------------------------
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)

# =========================
# 4. SCALING
# =========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =========================
# 5. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# =========================
# 6. MODELS (TUNED)
# =========================
models = {
    "Ridge": Ridge(alpha=10),
    "Lasso": Lasso(alpha=0.1),
    "Decision Tree": DecisionTreeRegressor(max_depth=10),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        random_state=42
    )
}

results = []

print("\n--- Improved Model Performance ---")

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    results.append([name, mae, mse, rmse, r2])

# Display results
results_df = pd.DataFrame(results, columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2 Score'])
print(results_df.to_string(index=False))


# =========================
# 7. RECOMMENDATION SYSTEM
# =========================
def get_recommendations(sku_id, top_n=10):

    # One-hot encoding on original dataset
    features = pd.get_dummies(df_rec[['category', 'subcategory']])

    # Cosine similarity
    cosine_sim = cosine_similarity(features, features)

    # Find index
    try:
        idx = df_rec.index[df_rec['sku_id'] == sku_id][0]
    except IndexError:
        return "SKU not found."

    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Top N
    sim_indices = [i[0] for i in sim_scores[1:top_n+1]]

    return df_rec[['sku_id', 'product_name', 'category', 'subcategory']].iloc[sim_indices]


# =========================
# 8. TEST RECOMMENDATION
# =========================
print("\n--- Top 10 Recommendations for SKU001000 ---")
print(get_recommendations('SKU001000'))

/Users/hetlimbani/Desktop/HET CLG/SEM 6/DL_LAB/dl_env/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)



--- Improved Model Performance ---
        Model       MAE          MSE       RMSE  R2 Score
        Ridge 77.205826 10142.676598 100.710856  0.966677
        Lasso 77.152006 10140.934216 100.702206  0.966683
Decision Tree 90.855381 15448.528994 124.292112  0.949245
Random Forest 78.917187 11058.798661 105.160823  0.963667

--- Top 10 Recommendations for SKU001000 ---
       sku_id                             product_name     category  \
1   SKU001001  PeakForm Premium Smartphones Silver 814  Electronics   
2   SKU001002     NextGen Classic Smartphones Pink 400  Electronics   
3   SKU001003    VitaPlus Elite Smartphones Silver 479  Electronics   
4   SKU001004    LunaGear Compact Smartphones Blue 801  Electronics   
5   SKU001005     SwiftMart Eco Smartphones Silver 371  Electronics   
6   SKU001006  FreshHub Classic Smartphones Silver 742  Electronics   
7   SKU001007      NextGen Elite Smartphones White 400  Electronics   
8   SKU001008    BrightPath Pro Smartphones Silver 482  Elec

In [20]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv('data/product_catalog.csv')

# Keep original copy for recommendation
df_rec = df.copy()

# =========================
# 2. DATA PREPROCESSING
# =========================

# ❌ DROP USELESS COLUMN (ALL NaN)
df.drop(['restock_days'], axis=1, inplace=True)

# -------------------------
# Handle Date
# -------------------------
df['launch_date'] = pd.to_datetime(df['launch_date'], format='%d/%m/%y', errors='coerce')
df['launch_year'] = df['launch_date'].dt.year

# Fill missing safely
df['launch_year'] = df['launch_year'].fillna(df['launch_year'].median())
df['launch_year'] = df['launch_year'].fillna(0)

df.drop(['launch_date'], axis=1, inplace=True)

# -------------------------
# Boolean to numeric
# -------------------------
df['is_active'] = df['is_active'].map({'TRUE': 1, 'FALSE': 0, True: 1, False: 0})
df['is_active'] = df['is_active'].fillna(0)

# -------------------------
# Tags → count
# -------------------------
def count_tags(tag_str):
    try:
        return len(ast.literal_eval(tag_str))
    except:
        return 0

df['tags_count'] = df['tags'].apply(count_tags)

# -------------------------
# Encode categorical columns
# -------------------------
cat_cols = ['category', 'subcategory', 'brand']

for col in cat_cols:
    df[col] = df[col].fillna("Unknown")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# =========================
# 3. FEATURE & TARGET
# =========================
X = df.drop(['sku_id', 'product_name', 'tags', 'current_price_usd'], axis=1)
y = df['current_price_usd']

# -------------------------
# FINAL NaN SAFETY (IMPORTANT)
# -------------------------
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)

# =========================
# 4. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 5. MODELS
# =========================
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(n_estimators=100)
}

results = []

print("\n--- Regression Model Performance ---")

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    results.append([name, mae, mse, rmse, r2])

# Display results
results_df = pd.DataFrame(results, columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2 Score'])
print(results_df.to_string(index=False))


# =========================
# 6. RECOMMENDATION SYSTEM
# =========================
def get_recommendations(sku_id, top_n=10):
    
    # One-hot encode category + subcategory
    features = pd.get_dummies(df_rec[['category', 'subcategory']])

    # Cosine similarity
    cosine_sim = cosine_similarity(features, features)

    # Find index
    try:
        idx = df_rec.index[df_rec['sku_id'] == sku_id][0]
    except IndexError:
        return "SKU not found."

    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Top N (excluding itself)
    sim_indices = [i[0] for i in sim_scores[1:top_n+1]]

    return df_rec[['sku_id', 'product_name', 'category', 'subcategory']].iloc[sim_indices]


# =========================
# 7. TEST RECOMMENDATION
# =========================
print("\n--- Top 10 Recommendations for SKU001000 ---")
print(get_recommendations('SKU001000'))

/Users/hetlimbani/Desktop/HET CLG/SEM 6/DL_LAB/dl_env/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)



--- Regression Model Performance ---
            Model       MAE          MSE       RMSE  R2 Score
Linear Regression 77.274728 10165.748617 100.825337  0.966601
            Ridge 77.204284 10140.046387 100.697797  0.966686
            Lasso 77.187370 10148.838798 100.741445  0.966657
    Decision Tree 98.200996 19269.890949 138.816033  0.936690
    Random Forest 78.240756 10658.638708 103.240683  0.964982

--- Top 10 Recommendations for SKU001000 ---
       sku_id                             product_name     category  \
1   SKU001001  PeakForm Premium Smartphones Silver 814  Electronics   
2   SKU001002     NextGen Classic Smartphones Pink 400  Electronics   
3   SKU001003    VitaPlus Elite Smartphones Silver 479  Electronics   
4   SKU001004    LunaGear Compact Smartphones Blue 801  Electronics   
5   SKU001005     SwiftMart Eco Smartphones Silver 371  Electronics   
6   SKU001006  FreshHub Classic Smartphones Silver 742  Electronics   
7   SKU001007      NextGen Elite Smartphones Wh